# Cross-View Cross-Encoder Scoring (CO PENALTY)

Giong baseline nhung bat penalty: empty_penalty (LAMBDA_PENALTY=2.0) + activity_loss (ALPHA_ACTIVITY=0.5). **Run-All** sweep 4 encoder x 4 fusion.

In [ ]:
import pandas as pd

# Đường dẫn file
match_path = "/kaggle/input/datasets/hunthi/en-vi-bendmark/merged_dataset/dataset/train/match/train_match.csv"
mismatch_path = "/kaggle/input/datasets/hunthi/en-vi-bendmark/merged_dataset/dataset/train/mismatch/train_mismatch.csv"

# Đọc dữ liệu
match_df = pd.read_csv(match_path)
mismatch_df = pd.read_csv(mismatch_path)

# In số lượng sample
print(f"Train Match     : {len(match_df):,} samples")
print(f"Train Mismatch  : {len(mismatch_df):,} samples")
print(f"Total Train     : {len(match_df) + len(mismatch_df):,} samples")

In [2]:
# Cai dat thu vien (Kaggle/Colab)
!pip install -q transformers sentencepiece einops

In [ ]:
import os, json, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [4]:
# =============================== CAU HINH ===============================
# Dataset da split (upload merged_dataset.zip lam Kaggle dataset slug: merged-cv-jd-scoring)
DATASET_BASE = "/kaggle/input/datasets/hunthi/en-vi-bendmark/merged_dataset"
CV_DIR     = os.path.join(DATASET_BASE, "cv")
JD_DIR     = os.path.join(DATASET_BASE, "jd")
SPLIT_ROOT = os.path.join(DATASET_BASE, "dataset")

WORK      = "/kaggle/working"
CACHE_DIR = os.path.join(WORK, "cache")
CKPT_DIR  = os.path.join(WORK, "checkpoints")
PLOT_DIR  = os.path.join(WORK, "plots")
RESULTS_CSV = os.path.join(WORK, "results_crossencoder_penalty.csv")
for d in (CACHE_DIR, CKPT_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

# 10 view CV, 5 view JD -> 50 cap view
CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities',
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']
NUM_PAIRS = len(CV_VIEWS) * len(JD_VIEWS)  # 50

# 4 encoder se duoc sweep tu dong (Run-All chay het, khong can sua tay)
ENCODERS = [
    "BAAI/bge-reranker-v2-m3",
    "jinaai/jina-reranker-v2-base-multilingual",
    "xlm-roberta-base",
    "bert-base-multilingual-cased",
]

MAX_LEN = 256
EPOCHS  = 150
BATCH   = 32

In [5]:
# =============================== HELPERS ===============================
def process_view(view_data):
    """missing_section -> [EMPTY_SECTION]; tra ve (text, exists_flag)."""
    if view_data is None:
        return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean:
            return "[EMPTY_SECTION]", 0
        return ". ".join(clean), 1
    val = str(view_data).lower()
    if val in ("missing_section", "", "none"):
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1


def load_json(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)


def read_split_rows(split):
    """Doc match + mismatch CSV cua 1 split -> list (cv, jd, score, bin_label)."""
    rows = []
    for label, b in (("match", 1), ("mismatch", 0)):
        p = os.path.join(SPLIT_ROOT, split, label, f"{split}_{label}.csv")
        df = pd.read_csv(p)
        for _, r in df.iterrows():
            rows.append((r["cv"], r["jd"], float(r["score"]), b))
    return rows


def compute_metrics(y_reg, y_bin, preds):
    preds = np.asarray(preds); y_reg = np.asarray(y_reg); y_bin = np.asarray(y_bin)
    pred_bin = (preds >= 0.5).astype(int)
    return {
        "MAE": mean_absolute_error(y_reg, preds),
        "MSE": mean_squared_error(y_reg, preds),
        "Accuracy": accuracy_score(y_bin, pred_bin),
        "Precision": precision_score(y_bin, pred_bin, zero_division=0),
        "Recall": recall_score(y_bin, pred_bin, zero_division=0),
        "F1-Score": f1_score(y_bin, pred_bin, zero_division=0),
    }


RESULT_COLS = ["Encoding Model", "Fusion", "MAE", "MSE",
               "Accuracy", "Precision", "Recall", "F1-Score"]


def append_result_csv(path, encoder, fusion, m):
    row = {"Encoding Model": encoder, "Fusion": fusion,
           "MAE": round(m["MAE"], 4), "MSE": round(m["MSE"], 4),
           "Accuracy": round(m["Accuracy"], 4), "Precision": round(m["Precision"], 4),
           "Recall": round(m["Recall"], 4), "F1-Score": round(m["F1-Score"], 4)}
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=RESULT_COLS)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(subset=["Encoding Model", "Fusion"], keep="last")
    df = df[RESULT_COLS]
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return df


def plot_history(hist, title, save_path):
    x = np.arange(1, len(hist["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(x, hist["train_loss"], label=f"Train Loss (final={hist['train_loss'][-1]:.4f})")
    ax1.plot(x, hist["val_loss"],   label=f"Val Loss (final={hist['val_loss'][-1]:.4f})")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} - Loss"); ax1.legend(); ax1.grid(True)
    ax2.plot(x, hist["train_acc"], label=f"Train Acc (final={hist['train_acc'][-1]:.4f})")
    ax2.plot(x, hist["val_acc"],   label=f"Val Acc (final={hist['val_acc'][-1]:.4f})")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{title} - Accuracy"); ax2.legend(); ax2.grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()

In [ ]:
# =============================== LOAD ENCODER ===============================
def load_encoder(name):
    """Tra ve (tokenizer, encoder_frozen, hidden_dim, slug).
    - bge-reranker-v2-m3: XLM-R-large fine-tune -> AutoModel nap base sach.
    - jina-reranker-v2 : uu tien custom code (eager attn); fallback XLM-R + bao cao key thieu.
    """
    slug = name.split("/")[-1]
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    enc = None
    if "jina" in name.lower():
        try:
            enc = AutoModel.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float32)
            print(f"[{slug}] loaded via AutoModel(trust_remote_code)")
        except Exception as e:
            print(f"[{slug}] AutoModel custom that bai -> fallback XLMRobertaModel: {e}")
            from transformers import XLMRobertaModel
            enc, info = XLMRobertaModel.from_pretrained(
                name, output_loading_info=True, torch_dtype=torch.float32)
            print(f"[{slug}] missing_keys={len(info['missing_keys'])} "
                  f"unexpected_keys={len(info['unexpected_keys'])} "
                  f"(neu core layer thieu nhieu -> feature jina khong dang tin)")
    else:
        enc = AutoModel.from_pretrained(name, trust_remote_code=True)
    enc = enc.to(device).eval()
    for p in enc.parameters():
        p.requires_grad_(False)
    hidden = enc.config.hidden_size
    with torch.no_grad():
        _t = tok(["test"], ["test"], return_tensors="pt", truncation=True, max_length=8).to(device)
        _o = enc(**_t)
    assert hasattr(_o, "last_hidden_state"), \
        f"{slug}: model khong tra ve last_hidden_state (co the nap nham classification head)"
    print(f"[{slug}] hidden_dim={hidden} | last_hidden_state OK")
    return tok, enc, hidden, slug

In [7]:
# =============== TRICH DAC TRUNG (CROSS-ENCODER: CLS cua cap [cv [SEP] jd]) ===============
@torch.no_grad()
def embed_pairs_cls(cv_texts, jd_texts, tok, enc):
    """50 cap (cv_view_i, jd_view_j) -> CLS (50, hidden)."""
    a = [cv_texts[i] for i in range(len(cv_texts)) for _ in range(len(jd_texts))]
    b = [jd_texts[j] for _ in range(len(cv_texts)) for j in range(len(jd_texts))]
    cls = []
    B = 32
    for i in range(0, len(a), B):
        t = tok(a[i:i+B], b[i:i+B], padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt").to(device)
        o = enc(**t)
        cls.append(o.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32))
    return np.concatenate(cls, 0)


def prepare_data(split, tok, enc, slug):
    """Moi mau -> CLS (50, hidden) + mask (50). Cache .npz theo slug+split."""
    cache = os.path.join(CACHE_DIR, f"{slug}_{split}_ce.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"  cache: {cache} | cls={d['cls'].shape}")
        return d["cls"], d["mask"], d["yreg"], d["ybin"]

    rows = read_split_rows(split)
    all_cls, all_mask, yreg, ybin = [], [], [], []
    for cvfn, jdfn, score, bvar in tqdm(rows, desc=f"[{slug}] ce {split}"):
        cv_obj = load_json(os.path.join(CV_DIR, cvfn)); cv_j = cv_obj.get("resume_structured", cv_obj)
        jd_obj = load_json(os.path.join(JD_DIR, jdfn)); jd_j = jd_obj.get("job_description_structured", jd_obj)
        cv_texts, cv_ex = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
        jd_texts, jd_ex = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])
        cls = embed_pairs_cls(list(cv_texts), list(jd_texts), tok, enc)   # (50, hidden)
        mask = np.outer(np.array(cv_ex, np.float32), np.array(jd_ex, np.float32)).flatten()
        all_cls.append(cls); all_mask.append(mask)
        yreg.append(score); ybin.append(bvar)

    cls_arr = np.asarray(all_cls, np.float32)     # (N, 50, hidden)
    mask_arr = np.asarray(all_mask, np.float32)   # (N, 50)
    yreg = np.asarray(yreg, np.float32); ybin = np.asarray(ybin, np.float32)
    np.savez_compressed(cache, cls=cls_arr, mask=mask_arr, yreg=yreg, ybin=ybin)
    return cls_arr, mask_arr, yreg, ybin

In [ ]:
# =============== DATASET + SCORER + 4 FUSION ===============
class CEDataset(Dataset):
    def __init__(self, cls, mask, yreg, ybin):
        self.cls = torch.tensor(cls, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.yreg = torch.tensor(yreg, dtype=torch.float32).view(-1, 1)
        self.ybin = torch.tensor(ybin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.cls)
    def __getitem__(self, i):
        return self.cls[i], self.mask[i], self.yreg[i], self.ybin[i]


class ViewPairScorer(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Linear(256, 1))
        nn.init.constant_(self.fc[2].bias, -2.0)
    def forward(self, cls):
        return torch.sigmoid(self.fc(cls)).squeeze(-1)


class WeightedSumFusion(nn.Module):
    def __init__(self, num_views=50):
        super().__init__(); self.layer = nn.Linear(num_views * 2, 1)
    def forward(self, a_ij, mask):
        return torch.sigmoid(self.layer(torch.cat([a_ij, mask], dim=-1)))


class FusionMLP(nn.Module):
    def __init__(self, num_views=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_views * 2, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        return self.net(torch.cat([a_ij, mask], dim=-1))


class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.ln = nn.LayerNorm(dim); self.fc = nn.Linear(dim, dim)
    def forward(self, x): return x + F.relu(self.fc(self.ln(x)))


class FusionResidualMLP(nn.Module):
    def __init__(self, num_views=50):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(num_views * 2, 256), nn.ReLU())
        self.res1, self.res2 = ResidualBlock(256), ResidualBlock(256)
        self.hidden = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
                                    nn.Linear(128, 64), nn.ReLU())
        self.out = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        x = self.inp(torch.cat([a_ij, mask], dim=-1))
        x = self.res1(x); x = self.res2(x); x = self.hidden(x)
        return self.out(x)


class FusionAttention(nn.Module):
    def __init__(self, num_views=50, d_model=64, nhead=8):
        super().__init__()
        self.proj = nn.Linear(2, d_model)   # 1 -> 2: nhan ca score va mask
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=256, dropout=0.2, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=3)
        self.head = nn.Sequential(nn.Linear(d_model * num_views, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        b = a_ij.shape[0]
        x = torch.stack([a_ij, mask], dim=-1)   # (B,50,2) - them mask vao input
        h = self.proj(x)
        h = self.attn(h)
        return self.head(h.reshape(b, -1))


class FusionWithScorer(nn.Module):
    def __init__(self, hidden, fusion):
        super().__init__(); self.scorer = ViewPairScorer(hidden); self.fusion = fusion
    def forward(self, cls, mask):
        raw = self.scorer(cls)          # (B,50)
        masked = raw * mask
        return self.fusion(masked, mask), masked, raw


FUSION_BUILDERS = {
    "weighted_sum": lambda: WeightedSumFusion(NUM_PAIRS),
    "mlp":          lambda: FusionMLP(NUM_PAIRS),
    "residual_mlp": lambda: FusionResidualMLP(NUM_PAIRS),
    "attention":    lambda: FusionAttention(NUM_PAIRS),
}

# Training + Evaluation

In [ ]:
# =============== TRAIN + EVAL (penalty + trainable task weights via softmax) ===============
LAMBDA_PENALTY     = 2.0
ALPHA_ACTIVITY     = 0.5
MARGIN_CONTRASTIVE = 0.5


def contrastive_loss(pred, ybin, margin=MARGIN_CONTRASTIVE):
    pos = ybin       * (1.0 - pred) ** 2
    neg = (1 - ybin) * F.relu(pred - margin) ** 2
    return (pos + neg).mean()


class TaskWeighter(nn.Module):
    """
    Hoc trong so dong cho 3 task loss: regression, classification, contrastive.
    Dung softmax de dam bao tong = 1 (weighted sum).
    1 dense layer -> 3 neuron -> softmax -> (w_reg, w_cls, w_contra)
    """
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 3)  
        nn.init.constant_(self.fc.weight, 0.0)
        with torch.no_grad():
            self.fc.bias.copy_(torch.tensor([1.5, 0.5, 0.0]))  

    def forward(self):
        dummy = torch.ones(1, 1, device=next(self.parameters()).device)
        logits = self.fc(dummy).squeeze(0)   # (3,)
        return torch.softmax(logits, dim=0)  # (w_reg, w_cls, w_contra), sum=1


def train_one(model, train_loader, val_loader, ckpt, lr=1e-3, start_epoch=0, epochs=None):
    if epochs is None: epochs = EPOCHS
    model      = model.to(device)
    task_w     = TaskWeighter().to(device)   # trainable task weights
    crit_reg   = nn.MSELoss(); crit_bin = nn.BCELoss()
    opt = optim.AdamW(
        list(model.parameters()) + list(task_w.parameters()),
        lr=lr, weight_decay=0.01
    )
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    hist  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    if start_epoch > 0 and os.path.exists(ckpt):
        ckpt_data = torch.load(ckpt, map_location=device)
        model.load_state_dict(ckpt_data["model"])
        task_w.load_state_dict(ckpt_data["task_w"])
        print(f"    Resume tu epoch {start_epoch} <- {ckpt}")

    best = float("inf")
    for ep in range(start_epoch, epochs):
        model.train(); task_w.train(); tl = 0.0; tp, tt = [], []
        for cls, mask, yr, yb in train_loader:
            cls, mask, yr, yb = cls.to(device), mask.to(device), yr.to(device), yb.to(device)
            opt.zero_grad()
            pred, masked, raw = model(cls, mask)

            # Lay trong so dong tu TaskWeighter
            w = task_w()   # (w_reg, w_cls, w_contra), sum=1

            l_reg   = crit_reg(pred, yr)
            l_cls   = crit_bin(pred, yb)
            l_contra = contrastive_loss(pred, yb)

            # Weighted sum voi trong so hoc duoc
            task = w[0] * l_reg + w[1] * l_cls + w[2] * l_contra

            empty_penalty = torch.mean(((1 - mask) * raw) ** 2)
            activity_loss = torch.mean(mask * (0.1 - masked).clamp(min=0))
            loss = (task
                    + LAMBDA_PENALTY * empty_penalty
                    + ALPHA_ACTIVITY  * activity_loss)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(task_w.parameters()), max_norm=1.0)
            opt.step(); tl += loss.item()
            tp += list((pred.detach().cpu().numpy().flatten() >= 0.5).astype(int))
            tt += list(yb.cpu().numpy().flatten().astype(int))

        model.eval(); task_w.eval(); vl = 0.0; vp, vt = [], []
        with torch.no_grad():
            w = task_w()  
            for cls, mask, yr, yb in val_loader:
                cls, mask, yr, yb = cls.to(device), mask.to(device), yr.to(device), yb.to(device)
                pred, masked, raw = model(cls, mask)
                v_task   = w[0] * crit_reg(pred, yr) + w[1] * crit_bin(pred, yb) \
                           + w[2] * contrastive_loss(pred, yb)
                v_pen    = torch.mean(((1 - mask) * raw) ** 2)
                v_act    = torch.mean(mask * (0.1 - masked).clamp(min=0))
                vl += (v_task + LAMBDA_PENALTY * v_pen + ALPHA_ACTIVITY * v_act).item()
                vp += list((pred.cpu().numpy().flatten() >= 0.5).astype(int))
                vt += list(yb.cpu().numpy().flatten().astype(int))

        av = vl / len(val_loader); sched.step(av)
        hist["train_loss"].append(tl / len(train_loader)); hist["val_loss"].append(av)
        hist["train_acc"].append(accuracy_score(tt, tp));  hist["val_acc"].append(accuracy_score(vt, vp))

        if (ep + 1) % 10 == 0:
            w_np = task_w().detach().cpu().numpy()
            print(f"    ep{ep+1:3d} | val_loss={av:.4f} | "
                  f"w_reg={w_np[0]:.3f} w_cls={w_np[1]:.3f} w_contra={w_np[2]:.3f}")

        if av < best:
            best = av
            # Luu ca model va task_w vao cung 1 checkpoint
            torch.save({
                "model":  model.state_dict(),
                "task_w": task_w.state_dict(),
            }, ckpt)

    return hist, best


def evaluate(model, test_loader, ckpt):
    model = model.to(device)
    ckpt_data = torch.load(ckpt, map_location=device)
    state = ckpt_data["model"] if "model" in ckpt_data else ckpt_data
    model.load_state_dict(state); model.eval()
    preds, yr, yb = [], [], []
    with torch.no_grad():
        for cls, mask, r, b in test_loader:
            pred, _, _ = model(cls.to(device), mask.to(device))
            preds += list(pred.cpu().numpy().flatten())
            yr    += list(r.numpy().flatten())
            yb    += list(b.numpy().flatten())
    return compute_metrics(yr, yb, preds)

In [ ]:
# =============== DRIVER: Run-All tu dong sweep 4 ENCODER x 4 FUSION ===============
SUFFIX = "_penalty"
ENCODERS_TO_RUN = ENCODERS
# ENCODERS_TO_RUN = [ENCODERS[0]]
# ENCODERS_TO_RUN = [ENCODERS[1]]
# ENCODERS_TO_RUN = [ENCODERS[2]]
# ENCODERS_TO_RUN = [ENCODERS[3]]

for ENCODER_NAME in ENCODERS_TO_RUN:
    print("\n" + "=" * 80)
    print("ENCODER:", ENCODER_NAME)
    print("=" * 80)
    try:
        tok, enc, hidden, slug = load_encoder(ENCODER_NAME)
        cls_tr, m_tr, y_tr, b_tr = prepare_data("train", tok, enc, slug)
        cls_v,  m_v,  y_v,  b_v  = prepare_data("val",   tok, enc, slug)
        cls_te, m_te, y_te, b_te = prepare_data("test",  tok, enc, slug)
        del enc, tok; gc.collect(); torch.cuda.empty_cache()

        train_loader = DataLoader(CEDataset(cls_tr, m_tr, y_tr, b_tr), batch_size=BATCH, shuffle=True)
        val_loader   = DataLoader(CEDataset(cls_v,  m_v,  y_v,  b_v),  batch_size=BATCH)
        test_loader  = DataLoader(CEDataset(cls_te, m_te, y_te, b_te), batch_size=BATCH)

        for FUSION_NAME, build in FUSION_BUILDERS.items():
            ckpt  = os.path.join(CKPT_DIR, f"{FUSION_NAME}_{slug}{SUFFIX}.pth")
            model = FusionWithScorer(hidden, build())
            hist, best_val = train_one(model, train_loader, val_loader, ckpt)
            plot_history(hist, f"{FUSION_NAME}{SUFFIX} | {slug}",
                         os.path.join(PLOT_DIR, f"{FUSION_NAME}{SUFFIX}-{slug}.png"))

            ckpt_data = torch.load(ckpt, map_location=device)
            if "task_w" in ckpt_data:
                task_w_final = TaskWeighter().to(device)
                task_w_final.load_state_dict(ckpt_data["task_w"])
                task_w_final.eval()
                with torch.no_grad():
                    w_final = task_w_final().cpu().numpy()
                lambda_str = (f"λ_reg={w_final[0]:.4f} "
                              f"λ_cls={w_final[1]:.4f} "
                              f"λ_contra={w_final[2]:.4f}")
            else:
                w_final = None
                lambda_str = "task_w not found"

            m = evaluate(model, test_loader, ckpt)
            if w_final is not None:
                m["λ_reg"]    = round(float(w_final[0]), 4)
                m["λ_cls"]    = round(float(w_final[1]), 4)
                m["λ_contra"] = round(float(w_final[2]), 4)

            append_result_csv(RESULTS_CSV, ENCODER_NAME, FUSION_NAME, m)
            print(f"  [{slug} | {FUSION_NAME}] best_val={best_val:.4f} "
                  f"MAE={m['MAE']:.4f} MSE={m['MSE']:.4f} "
                  f"Acc={m['Accuracy']:.4f} F1={m['F1-Score']:.4f} | {lambda_str}")

        del cls_tr, cls_v, cls_te; gc.collect()
    except Exception as e:
        import traceback
        print(f"!! BO QUA encoder {ENCODER_NAME} do loi: {e}")
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        continue

# Zip toan bo checkpoint sau khi train xong
import zipfile
zip_ckpt = os.path.join(WORK, "checkpoints_backup.zip")
with zipfile.ZipFile(zip_ckpt, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(CKPT_DIR):
        if fname.endswith(".pth"):
            zf.write(os.path.join(CKPT_DIR, fname), arcname=fname)
print(f"\nCheckpoints zipped: {zip_ckpt}")

print("\nXONG. Bang ket qua:")
res = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print("Luu tai:", RESULTS_CSV)
res

In [ ]:
# =============== BANG KET QUA TONG HOP ===============
res = pd.read_csv(RESULTS_CSV)
print("Luu tai:", RESULTS_CSV)
res

In [ ]:
!zip -r /kaggle/working/plots.zip /kaggle/working/plots

In [ ]:
# =============== INFERENCE 1 SAMPLE NGAU NHIEN TU TEST SET ===============
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# --- CAU HINH: doi dung encoder + fusion tot nhat ---
INFER_ENCODER = "bert-base-multilingual-cased"
INFER_FUSION  = "weighted_sum"
SUFFIX        = "_penalty"

slug     = INFER_ENCODER.split("/")[-1]
ckpt     = os.path.join(CKPT_DIR, f"{INFER_FUSION}_{slug}{SUFFIX}.pth")

# --- LOAD 1 SAMPLE NGAU NHIEN TU TEST SET ---
test_match    = pd.read_csv(os.path.join(SPLIT_ROOT, "test", "match",    "test_match.csv"))
test_mismatch = pd.read_csv(os.path.join(SPLIT_ROOT, "test", "mismatch", "test_mismatch.csv"))
test_all      = pd.concat([test_match, test_mismatch], ignore_index=True)
sample        = test_all.sample(1, random_state=random.randint(0, 9999)).iloc[0]

cvfn  = sample["cv"]
jdfn  = sample["jd"]
score = float(sample["score"])
true_label = 1 if score >= 0.5 else 0

print("=" * 70)
print(f"SAMPLE: cv={cvfn}  jd={jdfn}")
print(f"Ground-truth score={score:.4f}  label={'MATCH' if true_label==1 else 'MISMATCH'}")
print("=" * 70)

# --- LOAD JSON ---
cv_raw = load_json(os.path.join(CV_DIR, cvfn))
jd_raw = load_json(os.path.join(JD_DIR, jdfn))
cv_j   = cv_raw.get("resume_structured", cv_raw)
jd_j   = jd_raw.get("job_description_structured", jd_raw)

print("\n[CV VIEWS]")
for v in CV_VIEWS:
    text, exists = process_view(cv_j.get(v))
    status = "OK" if exists else "MISSING"
    preview = (text[:80] + "...") if text and len(text) > 80 else (text or "")
    print(f"  {v:<35} [{status}] {preview}")

print("\n[JD VIEWS]")
for v in JD_VIEWS:
    text, exists = process_view(jd_j.get(v))
    status = "OK" if exists else "MISSING"
    preview = (text[:80] + "...") if text and len(text) > 80 else (text or "")
    print(f"  {v:<35} [{status}] {preview}")

# --- ENCODE SAMPLE ---
print("\nEncoding sample...")
tok = AutoTokenizer.from_pretrained(INFER_ENCODER, trust_remote_code=True)
enc = AutoModel.from_pretrained(INFER_ENCODER, trust_remote_code=True).to(device).eval()

cv_texts, cv_ex = zip(*[process_view(cv_j.get(v)) for v in CV_VIEWS])
jd_texts, jd_ex = zip(*[process_view(jd_j.get(v)) for v in JD_VIEWS])

# embed_pairs_cls tao 50 cap (cv_i, jd_j)
cls_vec = embed_pairs_cls(list(cv_texts), list(jd_texts), tok, enc)   # (50, hidden)
mask_vec = np.outer(np.array(cv_ex, np.float32), np.array(jd_ex, np.float32)).flatten()
del enc, tok; gc.collect(); torch.cuda.empty_cache()

# --- LOAD MODEL VA PREDICT ---
hidden = cls_vec.shape[-1]
model  = FusionWithScorer(hidden, FUSION_BUILDERS[INFER_FUSION]())

ckpt_data  = torch.load(ckpt, map_location=device)
state_dict = ckpt_data["model"] if "model" in ckpt_data else ckpt_data
model.load_state_dict(state_dict)
model = model.to(device); model.eval()

cls_t  = torch.tensor(cls_vec,  dtype=torch.float32).unsqueeze(0).to(device)   # (1,50,H)
mask_t = torch.tensor(mask_vec, dtype=torch.float32).unsqueeze(0).to(device)   # (1,50)

with torch.no_grad():
    pred, masked, raw = model(cls_t, mask_t)

pred_score = pred.item()
pred_label = 1 if pred_score >= 0.5 else 0
raw_np     = raw.squeeze(0).cpu().numpy().reshape(10, 5)    # (10,5) alignment matrix
masked_np  = masked.squeeze(0).cpu().numpy().reshape(10, 5) # (10,5) masked matrix
mask_2d    = mask_vec.reshape(10, 5)

print("\n[INFERENCE RESULT]")
print(f"  Predicted score : {pred_score:.4f}")
print(f"  Predicted label : {'MATCH' if pred_label==1 else 'MISMATCH'}")
print(f"  Ground-truth    : {'MATCH' if true_label==1 else 'MISMATCH'} (score={score:.4f})")
print(f"  Correct         : {'YES ✓' if pred_label == true_label else 'NO ✗'}")

# --- VISUALIZATION ---
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# 1. Raw alignment matrix (truoc khi nhan mask)
ax = axes[0]
im = ax.imshow(raw_np, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(JD_VIEWS, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CV_VIEWS, fontsize=8)
ax.set_title("Raw Alignment Matrix S\n(before masking)", fontweight="bold")
plt.colorbar(im, ax=ax, fraction=0.046)
for i in range(10):
    for j in range(5):
        ax.text(j, i, f"{raw_np[i,j]:.2f}", ha="center", va="center",
                fontsize=7, color="black" if raw_np[i,j] < 0.7 else "white")

# 2. Masked alignment matrix (sau khi nhan mask)
ax = axes[1]
im2 = ax.imshow(masked_np, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(5)); ax.set_xticklabels(JD_VIEWS, rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CV_VIEWS, fontsize=8)
ax.set_title("Masked Alignment Matrix S'\n(after masking)", fontweight="bold")
plt.colorbar(im2, ax=axes[1], fraction=0.046)
for i in range(10):
    for j in range(5):
        val = masked_np[i,j]
        color = "gray" if mask_2d[i,j] == 0 else ("white" if val > 0.7 else "black")
        ax.text(j, i, "✗" if mask_2d[i,j] == 0 else f"{val:.2f}",
                ha="center", va="center", fontsize=7, color=color)

# 3. Summary bar
ax = axes[2]
cv_scores = masked_np.max(axis=1)   # max score cua moi CV view qua tat ca JD view
colors     = ["#0891B2" if s > 0.5 else "#F97316" for s in cv_scores]
bars = ax.barh(CV_VIEWS, cv_scores, color=colors, alpha=0.85)
ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="Threshold=0.5")
ax.set_xlim(0, 1.1)
ax.set_title(f"Max Score per CV View\n"
             f"Pred={pred_score:.4f} ({'MATCH' if pred_label==1 else 'MISMATCH'}) | "
             f"GT={score:.4f} ({'MATCH' if true_label==1 else 'MISMATCH'})",
             fontweight="bold")
ax.set_xlabel("Max Alignment Score")
ax.invert_yaxis()
ax.legend()
for bar, s in zip(bars, cv_scores):
    ax.text(s + 0.01, bar.get_y() + bar.get_height()/2,
            f"{s:.3f}", va="center", fontsize=8)

patch_match   = mpatches.Patch(color="#0891B2", label="Score > 0.5 (Match signal)")
patch_miss    = mpatches.Patch(color="#F97316", label="Score ≤ 0.5 (Weak signal)")
ax.legend(handles=[patch_match, patch_miss, plt.Line2D([0],[0], color="red",
          linestyle="--", label="Threshold=0.5")], fontsize=8, loc="lower right")

plt.suptitle(f"Inference Visualization | Encoder: {slug} | Fusion: {INFER_FUSION}\n"
             f"CV: {cvfn}  |  JD: {jdfn}", fontsize=11, fontweight="bold")
plt.tight_layout()
plot_path = os.path.join(PLOT_DIR, f"inference_{cvfn}_{jdfn}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nVisualization saved: {plot_path}")

In [ ]:
# =============== HIEN THI TAT CA PLOT LOSS / ACC TRONG NOTEBOOK ===============
import glob
from IPython.display import Image, display
pngs = sorted(glob.glob(os.path.join(PLOT_DIR, "*.png")))
print(f"Co {len(pngs)} plot loss/acc (moi anh: trai=Loss, phai=Accuracy):")
for p in pngs:
    print("-", os.path.basename(p))
    display(Image(filename=p))